In [553]:
import pandas as pd
import string
import nltk
import re
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC

from catboost import CatBoostClassifier

In [554]:
root_path = "/home/stefan/ioai-prep/kits/autor_versuri"
seed = 423

# Data

In [555]:
def clean_text(text):
    text = text.lower()
    text = re.sub("\n", " ", text)
    tok = nltk.word_tokenize(text)
    return " ".join(tok)

In [556]:
def extract_stylometric_features(df):
    # Syllable approximation (Romanian vowels: a, e, i, o, u, ă, â, î)
    vowels = "aeiouăâîAEIOUĂÂÎ"

    # 1. Structural features
    df["char_count"] = df["Versuri"].apply(len)
    df["word_count"] = df["Versuri"].apply(lambda x: len(x.split()))
    df["avg_word_len"] = df["char_count"] / df["word_count"]

    # 2. Punctuation density (Bacovia uses '...', Blaga uses metaphors/slashes)
    df["punct_count"] = df["Versuri"].apply(lambda x: len(re.findall(r"[.,!?;:]", x)))
    df["ellipsis_count"] = df["Versuri"].apply(lambda x: len(re.findall(r"\.\.\.", x)))

    # 3. Rhyme/Suffix features (Extract last 3 chars of each line)
    def get_endings(text):
        lines = text.split("\n")
        # If text is already flattened, this needs adjustment based on original format
        # If flattened, we use the last 3 chars of the whole quatrain
        return text[-3:].lower()

    df["suffix"] = df["Versuri"].apply(get_endings)

    # 4. Syllable count proxy
    df["syllables"] = df["Versuri"].apply(
        lambda x: sum(1 for char in x if char in vowels)
    )
    df["syllables_per_word"] = df["syllables"] / df["word_count"]

    return df

In [557]:
def prep_df(df: pd.DataFrame):
    df = df.drop(["Id", "Autor"], axis=1, errors='ignore')
    df["Versuri"] = df["Versuri"].apply(clean_text)
    extract_stylometric_features(df)
    return df

In [558]:
df = pd.read_csv(f"{root_path}/train.csv")
le = LabelEncoder()

train_df = prep_df(df)
y = le.fit_transform(df["Autor"])

train_df.head()

,Versuri,char_count,word_count,avg_word_len,punct_count,ellipsis_count,suffix,syllables,syllables_per_word
0,"pe barbari de-i risipeşte , ş-apoi vecinic pri...",103,17,6.058824,3,0,are,39,2.294118
1,sau va avea mulţi copii . pentru că totuna est...,100,22,4.545455,4,0,i .,38,1.727273
2,"copilăriei noastre . frate , ei urăsc cântecul...",70,13,5.384615,3,0,cul,27,2.076923
3,"veniţi lângă mine , tovarăşi ! că mâne-o să mo...",109,23,4.739130,4,0,eţi,38,1.652174
4,"iar tu , hyperion , rămâi oriunde ai apune ......",101,19,5.315789,6,1,e ?,39,2.052632


# Model

In [568]:
X_train, X_test, y_train, y_test = train_test_split(train_df, y, test_size=0.02, random_state=seed, stratify=y)

In [569]:
# char_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=25000)
# word_vectorizer = TfidfVectorizer(
#     analyzer="word", ngram_range=(1, 4), max_features=85000
# )

# vectorizer = FeatureUnion([("char", char_vectorizer), ("word", word_vectorizer)])

# X_train_tfidf = vectorizer.fit_transform(X_train["Versuri"])
# X_test_tfidf = vectorizer.transform(X_test["Versuri"])

In [599]:
pipeline = Pipeline([
    ('features', ColumnTransformer([
        # Branch 1: Word TF-IDF (Unigrams + Bigrams)
        ('word_tfidf', TfidfVectorizer(analyzer="word", ngram_range=(1, 4), max_features=85000), 'Versuri'),
        
        # Branch 2: Char TF-IDF (3-5 ngrams for roots/suffixes)
        ('char_tfidf', TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=25000), 'Versuri'),
        
        # Branch 3: Stylometric features
        ('stats', StandardScaler(), ['char_count', 'word_count', 'avg_word_len', 
                                     'punct_count', 'ellipsis_count', 'syllables', 'syllables_per_word'])
    ])),
    ('clf', LinearSVC(max_iter=20000, class_weight='balanced'))
    # ('clf', CatBoostClassifier(
    #     depth=10,
    #     iterations=300,
    #     loss_function="MultiClass",
    #     eval_metric="Accuracy",
    #     random_seed=seed,
    #     task_type="GPU",
    #     devices="0",
    # ))
])

In [600]:
def evaluate(model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(accuracy_score(y_test, pred))
    return model

In [601]:
evaluate(pipeline)

0.7971014492753623


Pipeline(steps=[('features',
                 ColumnTransformer(transformers=[('word_tfidf',
                                                  TfidfVectorizer(max_features=85000,
                                                                  ngram_range=(1,
                                                                               4)),
                                                  'Versuri'),
                                                 ('char_tfidf',
                                                  TfidfVectorizer(analyzer='char',
                                                                  max_features=25000,
                                                                  ngram_range=(2,
                                                                               4)),
                                                  'Versuri'),
                                                 ('stats', StandardScaler(),
                                                  ['char_count', 'word_count',
                                                   'avg_word_len',
                                                   'punct_count',
                                                   'ellipsis_count',
                                                   'syllables',
                                                   'syllables_per_word'])])),
                ('clf', LinearSVC(class_weight='balanced', max_iter=20000))])

# Submission

In [603]:
df_test_orig = pd.read_csv(f"{root_path}/test.csv")
df_test = prep_df(df_test_orig)

In [604]:
submission = df_test_orig[["Id"]]
submission["Autor"] = le.inverse_transform(pipeline.predict(df_test).ravel())

submission.head()

,Id,Autor
0,7oYYGUufTgpKPCcVUhTpAS,Grigore Vieru
1,EGWuQxkFSjW59anvGxW7nS,Grigore Vieru
2,XfHVBFedR9cDb4DBvuwtsF,Grigore Vieru
3,E3EbfZuVsZhUxqRYvBbywM,Lucian Blaga
4,kAxEyKrjhpvUu25CAjh5F6,Mihai Eminescu


In [605]:
submission.to_csv(f"{root_path}/submission.csv", index=False)